# WCB label transfer

Can labelled sentences from 24 other central banks stand in for scarce FOMC
labels? Three experiments, one model (roberta-large, winner config), all text
lowercased (WCB is lowercase-only -- casing must not masquerade as register).
Results land under corpus "twd-lc": comparisons are valid only inside this
lowercased room, against the lc control, never against the cased 0.707.

1. lc control: Shah train, lowercased -> prices the casing cost
2. transfer:   WCB only (zero FOMC labels) -> Shah test
3. augment:    Shah train + WCB           -> Shah test

### Colab Setup

In [ ]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C",s ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    # mounting is optional: without it, results land in the Colab session
    # filesystem and are lost when the runtime ends
    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception:
        print(f"Drive not mounted. Results will be written to {ROOT} "
              "and lost when the session ends.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Key Imports

In [10]:
import torch

from config import RESULTS_DIR, SHAH_PLM
from data.loader_wcb_labelled import fetch_annotated
from data.loader_twd_labelled import load_splits
from models.plm_finetune import finetune
from utils.results import already_done, save_result
import polars as pl

OUT = RESULTS_DIR / "results.csv"
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

results -> /content/drive/MyDrive/thesis/results.csv | device: cuda
NVIDIA A100-SXM4-40GB


### Targets

One row per target bank: its own labels, its held-out test split, and the pool of every other bank.

In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 78516
cfg = SHAH_PLM["roberta-large"]


def lc(df):
    return df.with_columns(pl.col("sentence").str.to_lowercase())


def matched(pool, n, seed):
    """Stratified sample of n rows, so class balance is preserved."""
    parts = []
    for lab, g in pool.groupby("label"):
        k = max(1, round(n * len(g) / len(pool)))
        parts.append(g.sample(n=min(k, len(g)), random_state=seed))
    return pd.concat(parts).sample(frac=1, random_state=seed)


# the loader drops WCB's own fomc rows, so the Fed enters only through TWD
wcb_full = fetch_annotated()
wcb_full = wcb_full.assign(
    sentence=wcb_full["sentence"].str.lower(), label=wcb_full["label_int"]
)[["bank_name", "sentence", "label"]]

# (tag, corpus, own train frame, test frame, borrowed pool)
TARGETS = []

tr, te = load_splits("benchmark", seed=SEED)
TARGETS.append(("fomc", "twd-lc", lc(tr).to_pandas(), lc(te).to_pandas(), wcb_full))

for tag, bank in [("ecb", "ecb"), ("boe", "bank_of_england"), ("boj", "bank_of_japan")]:
    own = wcb_full[wcb_full["bank_name"] == bank]
    o_tr, o_te = train_test_split(
        own, test_size=0.2, random_state=SEED, stratify=own["label"]
    )
    TARGETS.append((tag, f"{tag}-lc", o_tr, o_te, wcb_full[wcb_full["bank_name"] != bank]))

SIZES = {tag: (len(own_tr), len(pool)) for tag, _, own_tr, _, pool in TARGETS}
for tag, (n_own, n_pool) in SIZES.items():
    print(f"{tag:5s} own {n_own:>6,} | borrowed {n_pool:>7,}")


raw rows: 25000 | stance labels: {'neutral': 8737, 'dovish': 8312, 'hawkish': 7097, 'irrelevant': 854}
after dropping irrelevant, fomc and duplicates: 23151
Seed 78516 | Train: 1984 rows  |  Test: 496 rows  |  Total: 2480
fomc  own  1,984 | borrowed  23,151
ecb   own    780 | borrowed  22,175
boe   own    753 | borrowed  22,209
boj   own    763 | borrowed  22,197


### All four arms

Runs anything missing from `results.csv` and skips the rest.

In [12]:
# the Fed arms were named before the other banks existed; keep the old keys so
# finished runs still resolve as done
FOMC_KEYS = {
    "own": "roberta-large-lc",
    "borrowed": "wcb-only:roberta-large",
    "matched": "borrowed-matched:roberta-large",
    "union": "wcb-aug:roberta-large",
}
OTHER_KEYS = {
    "own": "own:roberta-large",
    "borrowed": "borrowed:roberta-large",
    "matched": "borrowed-matched:roberta-large",
    "union": "own+borrowed:roberta-large",
}

ARMS = [
    ("own", ("fomc", "ecb", "boe")),
    ("borrowed", ("fomc", "ecb", "boe")),
    ("matched", ("fomc", "ecb", "boe")),
    ("union", ("fomc", "ecb", "boe")),
]

for arm, tags in ARMS:
    for tag, corpus, own_tr, test, pool in TARGETS:
        if tag not in tags:
            continue
        key = (FOMC_KEYS if tag == "fomc" else OTHER_KEYS)[arm]
        if already_done(OUT, force=FORCE, model=key, corpus=corpus, seed=SEED):
            print(f"{corpus:7s} {key:30s} already done, skipping")
            continue

        if arm == "own":
            train = own_tr
        elif arm == "borrowed":
            train = pool
        elif arm == "matched":
            train = matched(pool, len(own_tr), SEED)
        else:
            train = pd.concat([own_tr, pool]).sample(frac=1, random_state=SEED)
        train = train[["sentence", "label"]]

        print(f"{corpus:7s} {key:30s} {len(train):,} training rows", flush=True)
        model, tok_, metrics = finetune(
            train,
            model_name=cfg["model_name"],
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            seed=SEED,
            test_df=test[["sentence", "label"]],
            device=DEVICE,
            verbose=True,
        )
        save_result(
            OUT,
            model=key,
            corpus=corpus,
            seed=SEED,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(metrics["test_macro_f1"], 4),
        )
        print(f"{corpus:7s} {key:30s} macro={metrics['test_macro_f1']:.4f}")
        del model, tok_
        torch.cuda.empty_cache()


twd-lc  roberta-large-lc               already done, skipping
ecb-lc  own:roberta-large              already done, skipping
boe-lc  own:roberta-large              already done, skipping
twd-lc  wcb-only:roberta-large         already done, skipping
ecb-lc  borrowed:roberta-large         already done, skipping
boe-lc  borrowed:roberta-large         already done, skipping
twd-lc  borrowed-matched:roberta-large already done, skipping
ecb-lc  borrowed-matched:roberta-large already done, skipping
boe-lc  borrowed-matched:roberta-large already done, skipping
twd-lc  wcb-aug:roberta-large          25,135 training rows


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  0: val CE=1.0980  acc=0.3764  wF1=0.2058  mF1=0.1823  es=0  230.4s
    epoch  1: val CE=1.0998  acc=0.3384  wF1=0.1711  mF1=0.1685  es=1  229.1s
    epoch  2: val CE=1.0990  acc=0.3384  wF1=0.1711  mF1=0.1685  es=2  229.3s
    epoch  3: val CE=1.0991  acc=0.3764  wF1=0.2058  mF1=0.1823  es=3  228.7s
    epoch  4: val CE=1.0988  acc=0.3384  wF1=0.1711  mF1=0.1685  es=4  228.4s
    epoch  5: val CE=1.0992  acc=0.2853  wF1=0.1266  mF1=0.1480  es=5  228.1s
    epoch  6: val CE=1.0986  acc=0.3764  wF1=0.2058  mF1=0.1823  es=6  228.6s
    epoch  7: val CE=1.1007  acc=0.2853  wF1=0.1266  mF1=0.1480  es=7  229.1s
twd-lc  wcb-aug:roberta-large          macro=0.2112
ecb-lc  own+borrowed:roberta-large     22,955 training rows


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    epoch  0: val CE=0.7181  acc=0.6981  wF1=0.6943  mF1=0.6957  es=0  207.7s
    epoch  1: val CE=0.6610  acc=0.7412  wF1=0.7414  mF1=0.7408  es=0  208.5s
    epoch  2: val CE=0.6669  acc=0.7421  wF1=0.7418  mF1=0.7411  es=0  207.3s
    epoch  3: val CE=0.7450  acc=0.7343  wF1=0.7336  mF1=0.7338  es=1  208.1s
    epoch  4: val CE=0.7707  acc=0.7327  wF1=0.7329  mF1=0.7324  es=2  206.5s


: 

### Table rows

In [ ]:
res = pd.read_csv(OUT)
ROWS = [
    ("Federal Reserve", "fomc", "twd-lc"),
    ("European Central Bank", "ecb", "ecb-lc"),
    ("Bank of England", "boe", "boe-lc"),
]


def f1(tag, corpus, arm):
    key = (FOMC_KEYS if tag == "fomc" else OTHER_KEYS)[arm]
    d = res[(res.model == key) & (res.corpus == corpus) & (res.seed == SEED)]
    return float(d["macro_f1"].iloc[0]) if len(d) else float("nan")


print(r"% ---- Table 11: own vs borrowed labels ----")
for name, tag, corpus in ROWS:
    n_own, n_pool = SIZES[tag]
    o, b = f1(tag, corpus, "own"), f1(tag, corpus, "borrowed")
    print(
        f"{name:<21} & {n_own:,} & {o:.3f} & {n_pool:,} & {b:.3f} & {b / o:.0%} \\\\"
    )

print()
print(r"% ---- Table 12: size-matched ----")
for name, tag, corpus in ROWS:
    n_own, _ = SIZES[tag]
    pool = next(p for t, _, _, _, p in TARGETS if t == tag)
    n_m = len(matched(pool, n_own, SEED))
    o, m = f1(tag, corpus, "own"), f1(tag, corpus, "matched")
    print(
        f"{name:<21} & {n_own:,} & {o:.3f} & {n_m:,} & {m:.3f} & {m / o:.0%} \\\\"
    )

print()
print(r"% ---- Table 13: own + borrowed ----")
for name, tag, corpus in ROWS:
    n_own, n_pool = SIZES[tag]
    o, b, u = (f1(tag, corpus, a) for a in ("own", "borrowed", "union"))
    print(
        f"{name:<21} & {o:.3f} & {b:.3f} & {n_own + n_pool:,} & {u:.3f} "
        f"& {u - max(o, b):+.3f} \\\\"
    )
